### Import Dependencies

In [1]:
import openai
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from langsmith import traceable, get_current_run_tree

load_dotenv("../../.env", override=True)

True

### Embedding function

In [2]:
@traceable(
    name="embed_query",
    run_type="embedding",
    metadata={"ls_provider": "openai", "ls_model_name": "text-embedding-3-small"},
)
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)

    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens,
            "total_tokens": response.usage.total_tokens,
        }

    return response.data[0].embedding

### Retrieval function

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

@traceable(
    name="retrieve_data",
    run_type="retriever",
)
def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings,
    }

### Format retrieved context function

In [4]:
@traceable(
    name="format_retrieved_context",
    run_type="prompt",
)
def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(
        context["retrieved_context_ids"],
        context["retrieved_context"],
        context["retrieved_context_ratings"],
    ):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context

### Create prompt template function

In [5]:
@traceable(
    name="build_prompt",
    run_type="prompt",
)
def build_prompt(preprocessed_context, query):
    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{query}
"""
    return prompt

### Generate answer function

In [6]:
@traceable(
    name="generate_answer",
    run_type="llm",
    metadata={"ls_provider": "openai", "ls_model_name": "gpt-5.4-nano"},
)
def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[{"role": "system", "content": prompt}],
        reasoning_effort="none",
    )

    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens,
        }

    return response.choices[0].message.content

### Combined RAG pipeline

In [7]:
@traceable(
    name="rag_pipeline",
)
def rag_pipeline(query, top_k=5):
    retrieved_context = retrieve_data(query, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, query)
    return generate_answer(prompt)

In [8]:
print(rag_pipeline("Do you have a USB connectable fan for hot summers?"))

Yes. We have a few USB-connectable fans suitable for hot summers:

1) Marame 120mm 5V USB Powered Fan with Speed Controller (ID: B0BRJS644Z, 4.7 rating)
- USB-powered with a 3.3 ft USB cable
- Adjustable speed switch: off / low / medium / high

2) HZD Desk Fan (Rechargeable wording, but it uses USB power cable) (ID: B0BXC72RLD, 4.3 rating)
- USB-powered with a 4.9 ft USB cable
- 3 speed settings: low / medium / high
- Quiet operation (noise claimed < 50 dB)

If you tell me whether you want a larger 120mm fan or a small personal desk fan, I can recommend the best fit.


In [9]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have above 4 rating.", 10))

Here are the earphones with above 4.0 rating from the available products:

1) B0CH8DRD6K (Rating 4.9) – Bluetooth 5.3 sports headband headphones with built-in ENC mic, for calls and workouts; 20+ hours playtime.

2) B0BRV544MV (Rating 4.4) – Bluetooth 5.3 wireless earbuds (40H total with charging case), deep bass, 4-mic clear calls, IPX7 waterproof.

If you want, tell me whether you prefer earbuds (in-ear) or headband/over-head style, and whether you need strong call quality or long battery life.


In [10]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have below 4 rating.", 10))

Here are some earphones from the available products that have a rating below 4.0:

1) pamu S29 Wireless Earbuds (Bluetooth 5.2, ANC/ENC) — Rating 3.9
2) RUNAR RNR1 Wireless Neckband Earphones (Bluetooth 5.0, sweat/rainproof, TF/SD compatible) — Rating 3.8
